# 39차 — 최신 모델 재비교 + MLflow 실험 추적 (TabPFN v2 · Chronos-2 vs V1-t)

> **명분**: `model_spec.md` §3의 DNN 재평가 트리거 ④("Chronos 계열 covariates 지원 릴리스") **발동** —
> Chronos-2(2025-10, 120M)가 known-future covariates를 지원. 함께 소표본(≤1만 행) 특화 파운데이션
> **TabPFN v2**(Nature 2025)를 후보에 추가. 작성: 2026-07-29
>
> **MLflow**: `mlflow_beginner_session1/2.ipynb`의 패턴(experiment → `start_run`(run_name·tags·description)
> → `log_params`/`log_metrics`/`log_model` → `search_runs` 비교 → **Model Registry 등록** → pyfunc 재로드)을
> 본 프로젝트 실험에 그대로 적용. 백엔드는 sqlite(`AI/mlruns/` — **gitignore, 로컬 전용**),
> 커밋 가능 지표는 sMAPE·상대값만(매출 절대액 금지 정책).

🔒 출력 제거 커밋 — 수치는 로컬 재실행으로 재현. env: `sajura-ag`(torch — tabpfn 2.0.9·chronos 2.3.1·mlflow).

**후보(사전 고정)** — ① `tabpfn_v2`: V1-t 구조에서 **회귀기만 교체**(같은 20열·비율 타깃·복원 — 가장 공정)
② `chronos2_cov`: 달력·기상을 known-future covariates로 전달, 1-step rolling teacher forcing
③ `chronos2_nocov`: covariates 제거 ablation(해석용, 판정 미사용).

**사전 등록 판정(M6.A9와 동일)**: 선택 fold 평균 MAE가 **V1-t 대비 −5% 이상 개선 시 채택 후보 격상**,
아니면 기록만. test fold(2026-04) 봉인 유지.

## 판정 요약 (TL;DR)

1. **두 후보 모두 기각(기록만) — V1-t 유지.** 선택 fold 평균 sMAPE:
   **V1-t 49.2**(MA-7 대비 −10.3%) < Chronos-2+cov 52.0(+13.9% vs V1-t) < Chronos-2 univ 53.3(+17.8%)
   < TabPFN v2 58.1(+29.0%). fold 승수도 각 2/5에 그침 — 사전 기준(−5% 개선) 미달.
2. **covariates 효과는 실재** — Chronos-2에서 cov 유/무 sMAPE 52.0 vs 53.3, 구 Chronos-bolt식
   regime 붕괴 없음(트리거 ④의 취지 자체는 유효했음). 그러나 zero-shot으로는 도메인 신호(학사·regime)를
   학습한 전용 소형 모델을 넘지 못함.
3. **TabPFN v2는 절반의 인상** — 평시 fold 4개에선 경쟁력(2025-11은 43.7로 전 모델 1위),
   **개강 fold(2026-03)에서 80.4로 붕괴** — 사전학습 분포 밖(extrapolation) 취약이 소표본 강점을 상쇄.
4. **실험 대장 구축** — 전 run이 MLflow(sqlite)에 기록되고, best run(V1-t)을 Model Registry
   `sajura-sales-forecaster`로 등록. 조회: `cd AI && mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`.
5. 재평가 트리거 ④는 **소진 처리**(Chronos-2로 검증 완료) — 남은 트리거: 데이터 2년+·다매장·drift.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow

warnings.filterwarnings("ignore")
_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp

# ── MLflow 설정 — 세션1 [12] 패턴 + sqlite 백엔드(mlflow 3.x 파일 저장소 유지보수 모드 대응) ──
mlflow.set_tracking_uri(f"sqlite:///{AI_DIR}/mlruns/mlflow.db")
experiment_name = "sajura_model_comparison"
mlflow.set_experiment(experiment_name)

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y, tx = ob.total_amount, ob.tx_count
folds = pp.make_monthly_folds(ob.index)
SEL = folds[:-1]
print("선택 fold:", [f["month"] for f in SEL], "| test 봉인:", folds[-1]["month"])

BEST = dict(learning_rate=0.0257, num_leaves=9, min_child_samples=10, subsample=0.7093,
            colsample_bytree=0.6796, reg_alpha=0.001, reg_lambda=0.0)
STATIC = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
          "is_post_renewal", "days_since_reopen"]

s = y.copy()
X1 = ob[STATIC].copy()
X1["lag_sales_h"] = s.shift(1)
X1["lag_tx_h"] = tx.shift(1)
r7 = s.shift(1).rolling(7).mean()
X1["roll7_h"] = r7
X1["roll_atv_h"] = (s / tx).shift(1).rolling(7).mean()
bydow = s.groupby(s.index.dayofweek)
X1["lag_dow"] = bydow.shift(1)
X1["roll4dow"] = bydow.apply(lambda g: g.shift(1).rolling(4).mean()).droplevel(0)
X1 = pd.concat([X1, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
b = X1.select_dtypes(bool).columns
X1[b] = X1[b].astype(int)
y_ratio = np.log1p(y) - np.log1p(r7)

# Chronos용 known-future covariates(달력·기상 — 예보로 미리 아는 값, lag 계열 제외)
COV = ob[STATIC].copy()
for i in range(7):
    COV[f"dow_{i}"] = (ob.index.dayofweek == i).astype(float)
COV = COV.astype(float).fillna(0.0)


def mae(a, p):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(p))))


def smape(a, p):
    a, p = np.asarray(a), np.asarray(p)
    return float(np.mean(2 * np.abs(a - p) / (a + np.abs(p))) * 100)


rows = []          # arm·fold별 지표 수집 → search_runs 밖에서도 표로 확인
NAIVE_MAE = float(np.mean([mae(y.loc[f["val"]], r7.loc[f["val"]]) for f in SEL]))


def log_arm(arm, fold_metrics, params, description, model=None, input_example=None):
    """세션1 [13] 패턴 — run 1개에 파라미터·지표(fold별 포함)·모델을 기록."""
    m_mae = float(np.mean([m["mae"] for m in fold_metrics]))
    metrics = {"smape_mean": float(np.mean([m["smape"] for m in fold_metrics])),
               "rel_mae_vs_ma7_pct": (m_mae / NAIVE_MAE - 1) * 100}
    if V1T_MAE is not None:
        metrics["rel_mae_vs_v1t_pct"] = (m_mae / V1T_MAE - 1) * 100
    with mlflow.start_run(run_name=arm,
                          tags={"model": arm, "dataset": "pilot-store", "step": "modern-2026"},
                          description=description):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for m in fold_metrics:
            mlflow.log_metric(f"smape_{m['fold'].replace('-', '_')}", m["smape"])
        if model is not None:
            mlflow.lightgbm.log_model(model, name="model", input_example=input_example)  # 세션2 [15] 전용 flavor 패턴
    for m in fold_metrics:
        rows.append({"arm": arm, **m})
    return m_mae


V1T_MAE = None

In [ ]:
# ── ① V1-t 참조 run — 기존 프로토콜 재현 (세션1 [13] 패턴으로 기록) ──
fm = []
last_model = None
for f in SEL:
    tr, va = f["train"], f["val"]
    ytr, yva = y_ratio.loc[tr], y_ratio.loc[va]
    k = ytr.notna()
    m = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1, objective="l2", **BEST)
    m.fit(X1.loc[tr][k], ytr[k], eval_set=[(X1.loc[va], yva)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = np.expm1(m.predict(X1.loc[va]) + np.log1p(r7.loc[va]))
    fm.append(dict(fold=f["month"], smape=smape(y.loc[va], p), mae=mae(y.loc[va], p)))
    last_model = m

V1T_MAE = log_arm("v1t_lgbm", fm,
                  params={**BEST, "target": "log1p(y)-log1p(roll7)", "protocol": "walk-forward 5 folds"},
                  description="현행 V1-t — LightGBM 비율 타깃 하이브리드 (05 모델 카드)",
                  model=last_model, input_example=X1.loc[SEL[-1]["val"]].head(2))
print(f"v1t_lgbm 기록 완료 — sMAPE {np.mean([x['smape'] for x in fm]):.2f}")

In [ ]:
# ── ② TabPFN v2 — 회귀기만 교체 (같은 20열·비율 타깃) ──
from tabpfn import TabPFNRegressor

t0 = time.time()
fm = []
for f in SEL:
    tr, va = f["train"], f["val"]
    ytr = y_ratio.loc[tr]
    k = ytr.notna()
    reg = TabPFNRegressor(random_state=42)
    reg.fit(X1.loc[tr][k].values.astype(np.float32), ytr[k].values.astype(np.float32))
    p = np.expm1(reg.predict(X1.loc[va].values.astype(np.float32)) + np.log1p(r7.loc[va]))
    fm.append(dict(fold=f["month"], smape=smape(y.loc[va], p), mae=mae(y.loc[va], p)))
    print(f"  {f['month']} 완료 ({time.time()-t0:.0f}s)")

log_arm("tabpfn_v2", fm,
        params={"model_detail": "tabpfn 2.0.9 (TabPFN-v2 regressor ckpt, Nature 2025)",
                "mode": "V1-t 구조에서 회귀기 교체", "n_estimators": "default"},
        description="소표본 특화 tabular 파운데이션 — 가중치 대형이라 log_model 생략")
print(f"tabpfn_v2 기록 완료 — sMAPE {np.mean([x['smape'] for x in fm]):.2f}")

In [ ]:
# ── ③ Chronos-2 — covariates 유/무, 1-step rolling teacher forcing (fold별 배치) ──
import torch
from chronos import Chronos2Pipeline

torch.manual_seed(42)
pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
open_idx = list(ob.index)
pos = {d: i for i, d in enumerate(open_idx)}
yv_all = y.values.astype(np.float32)

for use_cov in (True, False):
    arm = "chronos2_cov" if use_cov else "chronos2_nocov"
    t0 = time.time()
    fm = []
    for f in SEL:
        inputs = []
        for d in f["val"]:
            i = pos[d]
            item = {"target": yv_all[:i]}
            if use_cov:
                item["past_covariates"] = {c: COV[c].values[:i].astype(np.float32) for c in COV}
                item["future_covariates"] = {c: COV[c].values[i:i + 1].astype(np.float32) for c in COV}
            inputs.append(item)
        q, _ = pipe.predict_quantiles(inputs, prediction_length=1, quantile_levels=[0.5])
        preds = np.array([float(t.squeeze()) for t in q])
        fm.append(dict(fold=f["month"], smape=smape(y.loc[f["val"]], preds),
                       mae=mae(y.loc[f["val"]], preds)))
        print(f"  {arm} {f['month']} 완료 ({time.time()-t0:.0f}s)")
    log_arm(arm, fm,
            params={"model_detail": "amazon/chronos-2 (chronos-forecasting 2.3.1)",
                    "covariates": str(use_cov), "mode": "1-step rolling teacher forcing"},
            description="시계열 파운데이션 zero-shot — 재평가 트리거 ④ 검증" if use_cov
                        else "covariates 제거 ablation (판정 미사용)")
    print(f"{arm} 기록 완료")

In [ ]:
# ── 비교 — 세션2 [9][10] 패턴: search_runs → 정렬 표 + 사전 등록 판정 ──
experiment = mlflow.get_experiment_by_name(experiment_name)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs_df = runs_df[runs_df.status == "FINISHED"]  # 재실행 시 실패 run 제외

columns_to_show = ["run_id", "tags.mlflow.runName", "metrics.smape_mean",
                   "metrics.rel_mae_vs_ma7_pct", "metrics.rel_mae_vs_v1t_pct", "start_time"]
board = runs_df[columns_to_show].sort_values("metrics.smape_mean")
display(board.head(10))

res = pd.DataFrame(rows)
pv = res.pivot_table(index="fold", columns="arm", values="smape").round(2)
display(pv)

print("===== 판정 (사전 등록: V1-t 대비 MAE −5% 이상 개선 시 격상) =====")
agg_mae = res.groupby("arm").mae.mean()
for arm in ("tabpfn_v2", "chronos2_cov"):
    d = (agg_mae[arm] / agg_mae["v1t_lgbm"] - 1) * 100
    wins = int((pv[arm] < pv["v1t_lgbm"]).sum())
    print(f"{arm}: V1-t 대비 {d:+.1f}% | fold 승수 {wins}/5 → {'격상' if d <= -5 else '기각(기록만)'}")

In [ ]:
# ── Registry — 세션2 [17][19][21] 패턴: best run 등록 + pyfunc 재로드 검증 ──
best_run = runs_df.sort_values("metrics.smape_mean").iloc[0]
best_run_id, best_run_name = best_run["run_id"], best_run["tags.mlflow.runName"]
print("best run:", best_run_name, "| sMAPE", round(best_run["metrics.smape_mean"], 2))
assert best_run_name == "v1t_lgbm", "사전 기준상 V1-t가 최상이어야 함 — 아니면 판정 재검토"

registered = mlflow.register_model(model_uri=f"runs:/{best_run_id}/model",
                                   name="sajura-sales-forecaster",
                                   tags={"task": "daily-sales-forecast", "stage": "record-only"})
loaded = mlflow.pyfunc.load_model(f"models:/sajura-sales-forecaster/{registered.version}")
va = SEL[-1]["val"]
check = np.expm1(loaded.predict(X1.loc[va].head(3)) + np.log1p(r7.loc[va].head(3)))
print(f"registry v{registered.version} 재로드 검증 — 예측 {len(check)}건 정상 (finite: {np.isfinite(check).all()})")
print("※ 등록은 기록용 — 서빙은 stateless fit-on-request(predictor.py)라 registry를 참조하지 않는다.")

### 관찰 — 세부

- **V1-t 우위의 구조**: fold별로 보면 후보들이 평시 fold에선 대등~우세인 경우도 있으나
  (TabPFN 2025-11 43.7 = 전 모델 최고), **개강 fold(2026-03)에서 전부 무너짐**(V1-t 35.6 vs
  TabPFN 80.4·Chronos-2 50.4). 도메인 신호(학사 주차·regime)를 지역 데이터로 학습한 전용 소형
  모델의 우위가 파운데이션 모델의 일반 지식을 이김 — M6.A9 결론이 최신 세대에서도 반복.
- **Chronos-2의 진보는 인정** — covariates가 실제로 작동(cov 52.0 vs nocov 53.3)하고 bolt식
  붕괴가 사라짐. 다매장·cold-start 국면에선 여전히 관심 후보(10 §4 global prior와 비교 대상).
- **TabPFN의 교훈**: 소표본 강점은 실재하나 분포 밖 외삽(regime·개강)에 취약 — 우리 문제의
  난점이 "표본 수"가 아니라 "비정상성(non-stationarity)"임을 재확인.
- **MLflow 도입 판단**: 실험 추적·비교 UI·Registry가 수업 패턴 그대로 재현됨. 단 서빙이
  stateless라 Registry는 기록용 — 배포 연동은 M7.A5 [2단계](주간 재학습 도입 시) 재검토.
- 한계: Chronos-2·TabPFN 모두 zero-shot/기본 설정 — fine-tuning은 미검증(비용 대비 우선순위 낮음,
  데이터 2년+ 축적 시 재평가 트리거와 함께).

### 다음 단계

- spec 반영(docs PR): model_spec §3 DNN 보류 항목에 "트리거 ④ 소진(Chronos-2 검증 완료, 39차)" 주석.
- `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db` (AI/에서) — 발표용 스크린샷 소스.